In [ ]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rc("font", family="Microsoft YaHei")
matplotlib.rcParams["axes.unicode_minus"] = False

import sys
import torch
import numpy as np
import xarray as xr
from pathlib import Path

model_dir = Path("../hls-foundation-os/pretrained_models/prithvi_100m")
sys.path.insert(0, str(model_dir))

print("库导入成功")

In [ ]:
import torch
import yaml
from prithvi_mae import PrithviMAE

with open(model_dir / "config.yaml") as f:
    cfg = yaml.safe_load(f)

ckpt = torch.load(
    model_dir / "Prithvi_100M.pt",
    map_location="cpu"
)
state_dict = ckpt

embed_dim         = state_dict["encoder.norm.weight"].shape[0]
decoder_embed_dim = state_dict["decoder.decoder_embed.bias"].shape[0]
num_heads         = embed_dim // 64

model = PrithviMAE(
    img_size=224, patch_size=16, num_frames=3,
    tubelet_size=1, in_chans=6,
    embed_dim=embed_dim, depth=12, num_heads=num_heads,
    decoder_embed_dim=decoder_embed_dim,
    decoder_depth=8, decoder_num_heads=16,
    mlp_ratio=4.0, norm_pix_loss=False,
)
model.load_state_dict(state_dict, strict=False)
model.eval()
print(f"模型加载成功，参数量: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

In [ ]:
ds = xr.open_dataset("../data/processed/sydney_s2_2023.nc")

MEAN = [343.4, 546.8, 444.1, 2942.5, 1444.6, 899.7]
STD  = [255.0, 340.6, 373.0, 1232.4,  852.0, 680.1]
BANDS = ["nbart_blue","nbart_green","nbart_red",
         "nbart_nir_2","nbart_swir_2","nbart_swir_3"]

# 从影像中心裁剪一个 224×224 的 patch
H = ds.sizes["y"]
W = ds.sizes["x"]
cy, cx = H // 2, W // 2
P = 224

arrays = []
for i, band in enumerate(BANDS):
    arr = ds[band].isel(
        time=0,
        y=slice(cy-P//2, cy+P//2),
        x=slice(cx-P//2, cx+P//2),
    ).values.astype(np.float32)
    arr = np.nan_to_num(arr, nan=0.0)
    arr = (arr - MEAN[i]) / STD[i]
    arrays.append(arr)

# 堆叠为 (C, H, W) 再扩展为 (B, C, T, H, W)
image = np.stack(arrays, axis=0)               # (6, 224, 224)
tensor = torch.tensor(image, dtype=torch.float32)
tensor = tensor.unsqueeze(0).unsqueeze(2)      # (1, 6, 1, 224, 224)

# Prithvi 需要 T=3，复制时间步
tensor = tensor.repeat(1, 1, 3, 1, 1)         # (1, 6, 3, 224, 224)
print(f"输入 tensor shape: {tensor.shape}")

In [ ]:
import rasterio
import numpy as np
import torch
from pathlib import Path

data_dir = Path("../data/raw/hls_sample")

BANDS = ["B02", "B03", "B04", "B8A", "B11", "B12"]
MEAN  = [343.4, 546.8, 444.1, 2942.5, 1444.6, 899.7]
STD   = [255.0, 340.6, 373.0, 1232.4,  852.0, 680.1]

arrays = []
for i, band in enumerate(BANDS):
    with rasterio.open(data_dir / f"{band}.tif") as src:
        arr = src.read(1).astype(np.float32)
        nodata = src.nodata
        if nodata is not None:
            arr[arr == nodata] = np.nan

    arr = np.nan_to_num(arr, nan=0.0)

    # 从中心裁剪 224×224
    H, W = arr.shape
    cy, cx = H // 2, W // 2
    arr = arr[cy-112:cy+112, cx-112:cx+112]

    arr = (arr - MEAN[i]) / STD[i]
    arrays.append(arr)

image  = np.stack(arrays, axis=0)               # (6, 224, 224)
tensor = torch.tensor(image, dtype=torch.float32)
tensor = tensor.unsqueeze(0).unsqueeze(2)        # (1, 6, 1, 224, 224)
tensor = tensor.repeat(1, 1, 3, 1, 1)           # (1, 6, 3, 224, 224)

print(f"输入 tensor shape: {tensor.shape}")
print(f"值范围: min={tensor.min():.3f}  max={tensor.max():.3f}")